# MedHELM Workshop - Running a Tiny Evaluation

In this notebook, you'll learn how to:
1. Set up environment variables for a HELM run
2. Execute a tiny evaluation (10 instances)
3. Understand the output structure
4. View and interpret results

This follows the MedHELM quickstart guide from the official documentation.

## 1. Configuration

First, let's set up the configuration for our tiny evaluation.

In [ ]:
import os

# Define paths and parameters
HOME_DIR = os.path.expanduser("~")
OUTPUT_PATH = os.path.join(HOME_DIR, "benchmark_output")
SUITE_NAME = "tiny-medhelm-workshop"
MAX_EVAL_INSTANCES = 10

# For this example, we'll use PubMedQA scenario with a small model
# Note: You may need to adjust the model deployment based on your setup
RUN_ENTRIES = "pubmed_qa:model=openai/gpt-3.5-turbo"

# Alternatively, if you have local models or HuggingFace access:
# RUN_ENTRIES = "pubmed_qa:model=qwen/qwen2.5-7b-instruct,model_deployment=huggingface/qwen2.5-7b-instruct"

print("Configuration:")
print(f"  Output Path: {OUTPUT_PATH}")
print(f"  Suite Name: {SUITE_NAME}")
print(f"  Max Instances: {MAX_EVAL_INSTANCES}")
print(f"  Run Entries: {RUN_ENTRIES}")

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f"\n✓ Output directory ready")

## 2. Understanding the PubMedQA Scenario

**PubMedQA** is a question-answering dataset based on PubMed abstracts. It tests a model's ability to answer medical research questions.

- **Task Type**: Binary QA (yes/no/maybe)
- **Domain**: Medical research literature
- **Evaluation**: Exact match accuracy
- **Dataset Size**: ~1,000 questions (we'll use 10 for this tiny evaluation)

## 3. Running the Evaluation

Now let's run the evaluation using the `helm-run` command.

**Note**: This may take several minutes depending on the model and your configuration.

In [ ]:
import subprocess

# Build the helm-run command
cmd = [
    "helm-run",
    "--run-entries", RUN_ENTRIES,
    "--suite", SUITE_NAME,
    "--max-eval-instances", str(MAX_EVAL_INSTANCES),
    "--output-path", OUTPUT_PATH
]

print("Executing command:")
print(" ".join(cmd))
print("\n" + "="*60)
print("Running evaluation... This may take a few minutes.")
print("="*60 + "\n")

try:
    # Run the command and capture output
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=600  # 10 minute timeout
    )
    
    print("STDOUT:")
    print(result.stdout)
    
    if result.stderr:
        print("\nSTDERR:")
        print(result.stderr)
    
    if result.returncode == 0:
        print("\n✓ Evaluation completed successfully!")
    else:
        print(f"\n✗ Evaluation failed with return code {result.returncode}")
        
except subprocess.TimeoutExpired:
    print("\n✗ Evaluation timed out after 10 minutes")
except FileNotFoundError:
    print("\n✗ helm-run command not found. Make sure the environment is activated.")
    print("Try running: source ~/medhelm-env/bin/activate")
except Exception as e:
    print(f"\n✗ Error: {str(e)}")

## 4. Examining the Output Structure

Let's explore what files were created during the evaluation.

In [ ]:
import os
from pathlib import Path

runs_path = os.path.join(OUTPUT_PATH, "runs", SUITE_NAME)

if os.path.exists(runs_path):
    print(f"Output directory structure for suite '{SUITE_NAME}':")
    print("\n" + runs_path)
    
    # Walk through directory tree
    for root, dirs, files in os.walk(runs_path):
        level = root.replace(runs_path, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}├── {os.path.basename(root)}/")
        subindent = ' ' * 2 * (level + 1)
        for file in files[:10]:  # Limit to first 10 files per directory
            print(f"{subindent}├── {file}")
        if len(files) > 10:
            print(f"{subindent}└── ... and {len(files) - 10} more files")
        if level > 2:  # Limit depth
            break
else:
    print(f"✗ Output directory not found: {runs_path}")
    print("The evaluation may not have completed successfully.")

## 5. Viewing Results

Let's look at the results JSON file to see how the model performed.

In [ ]:
import json
import glob

# Find scenario_state.json files
state_files = glob.glob(os.path.join(runs_path, "**", "scenario_state.json"), recursive=True)

if state_files:
    print(f"Found {len(state_files)} scenario state file(s)\n")
    
    for state_file in state_files[:1]:  # Look at first one
        print(f"Examining: {state_file}")
        print("=" * 60)
        
        with open(state_file, 'r') as f:
            data = json.load(f)
        
        # Show key information
        if 'request_states' in data:
            num_requests = len(data['request_states'])
            print(f"\nNumber of requests: {num_requests}")
            
            if num_requests > 0:
                print("\nFirst request example:")
                first_request = data['request_states'][0]
                
                if 'request' in first_request:
                    request = first_request['request']
                    print(f"  Prompt (first 200 chars): {request.get('prompt', '')[:200]}...")
                
                if 'result' in first_request:
                    result = first_request['result']
                    print(f"  Completions: {len(result.get('completions', []))}")
                    if result.get('completions'):
                        print(f"  First completion: {result['completions'][0].get('text', '')}")
else:
    print("✗ No scenario state files found")

## 6. Creating a Summary

Now let's use `helm-summarize` to create a summary of our results.

In [ ]:
import subprocess

RELEASE_NAME = "tiny-workshop-release"

# Download the MedHELM schema if not already present
schema_path = os.path.join(OUTPUT_PATH, "schema_medhelm.yaml")
if not os.path.exists(schema_path):
    print("Downloading MedHELM schema...")
    subprocess.run([
        "wget",
        "-O", schema_path,
        "https://raw.githubusercontent.com/stanford-crfm/helm/v0.5.7/src/helm/benchmark/static/schema_medhelm.yaml"
    ])

# Run helm-summarize
cmd = [
    "helm-summarize",
    "--suite", SUITE_NAME,
    "--schema", schema_path,
    "--release", RELEASE_NAME,
    "--output-path", OUTPUT_PATH
]

print("Creating summary...")
print(" ".join(cmd))
print()

try:
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    
    if result.returncode == 0:
        print("\n✓ Summary created successfully!")
    else:
        print(f"\n✗ Summarization failed with return code {result.returncode}")
except Exception as e:
    print(f"\n✗ Error: {str(e)}")

## 7. Launching the Local Leaderboard

Finally, let's launch a local web server to view the results in a browser-friendly format.

**Note**: The helm-server command will block execution. You'll need to stop it (Kernel -> Interrupt) when done viewing.

In [ ]:
print("To launch the leaderboard server, run this command in a terminal:")
print()
print(f"helm-server --release {RELEASE_NAME} --output-path {OUTPUT_PATH}")
print()
print("The server will print a URL where you can view the leaderboard.")
print("Press Ctrl+C in the terminal to stop the server when done.")

## Summary

In this notebook, you:
1. ✓ Configured a tiny evaluation with 10 instances
2. ✓ Ran the evaluation on the PubMedQA scenario
3. ✓ Examined the output structure
4. ✓ Created a summary of results
5. ✓ Learned how to launch the local leaderboard

### Next Steps:

- Explore different scenarios in `workshop-notebook-3-scenarios.ipynb`
- Run larger evaluations with more instances
- Compare multiple models
- Create custom benchmarks in `workshop-notebook-4-custom-benchmarks.ipynb`